**Tools used:** SQL (SQLite), Python (pandas), Power BI  
**Role:** Data Analyst  
**Stakeholder:** Sales & Distribution Management  

# Customer Cohort Analysis – Digital Adoption

## 1. Context & Business Problem

The company recently launched a digital ordering application aimed at migrating traditional retail customers toward a fully digital sales model. While adoption has increased since launch, a significant portion of registered customers remain inactive or continue placing orders through non-digital channels.

Management has set a long-term objective to achieve 100% digital ordering adoption. However, limited visibility exists regarding which customer segments have adopted the platform, which remain resistant, and how digital adoption varies across routes.

This analysis aims to understand customer adoption behavior, identify non-digital and hybrid customers, and provide insights to support data-driven decision-making around digital transformation initiatives.


## 2. Data Sources

The analysis is based on anonymized datasets extracted from existing Power BI dashboards used by the sales and distribution teams.

The primary datasets include:

- **Customer Master Data:** Contains customer identifiers, route assignment, and current digital adoption status.
- **Application Registration Data:** Contains records of customers registered in the digital ordering application.

## 3. Analytical Objectives

- Identify customers registered in the digital platform versus those not registered.
- Segment customers into digital, hybrid, and non-digital cohorts.
- Analyze adoption patterns by route.
- Detect gaps between registration and active digital usage.
- Generate actionable insights to support future qualitative research and strategic initiatives.


## 4. SQL Analysis

This fase has de purpose to understand Who use the app?, Where are there? and How so far are there from de digital adoption?, using strutured data.

In [1]:
#Se importan las librerias necesarias

import pandas as pd
import sqlite3

In [2]:
#Se leen los datasets correspondientes
registros_df=pd.read_csv(r"../Data_processed_anon/Registros_Clientes_clean.csv")
status_df=pd.read_csv(r"../Data_processed_anon/Fully_digital_clean.csv")

In [3]:
#Se crea la base de datos SQLite3
conn = sqlite3.connect(r"../Data_processed_anon/digital_adoption.db")

In [4]:
#Se exportan datasets a tablas sql

registros_df.to_sql(
    name="registros",
    con = conn,
    if_exists="replace",
    index=False
)

status_df.to_sql(
    name="status",
    con=conn,
    if_exists="replace",
    index=False
)

1789

In [5]:
#Se verifican que las tablas existan
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,registros
1,status


In [6]:
#Querys de prueba 
query_prueba= """
        SELECT *
        FROM registros
        LIMIT 5;
"""

pd.read_sql(query_prueba,conn)


,distrito_sap,ruta,nip_cliente,nombre_cliente,status_registro
0,4210,RT020,200001743,Super Buena Vista 31,no registrado
1,4210,RT004,200001258,Comercial El Encanto 69,no registrado
2,4210,RT008,200001737,Frutería El Sol 16,no registrado
3,4210,RT015,200000179,Abarrotes Del Norte 98,no registrado
4,4210,RT002,200001649,Abarrotes Del Valle 45,registrado


In [7]:
query_prueba= """
        SELECT *
        FROM status
        LIMIT 5;
"""

pd.read_sql(query_prueba,conn)

,región,división,distrito,cedis,ruta_sap,cliente_id,nombre_cliente,categoría,%_venta_digital,venta_digital
0,NORESTE,DIV MONTERREY,4210,150 SAN NICOLAS DE LOS GARZA,RT001,200000018,Abarrotes El Progreso 72,non digital,0 %,$0
1,NORESTE,DIV MONTERREY,4210,150 SAN NICOLAS DE LOS GARZA,RT001,200000324,Minisuper La Esperanza 85,non digital,0 %,$0
2,NORESTE,DIV MONTERREY,4210,150 SAN NICOLAS DE LOS GARZA,RT001,200000612,Depósito La Fortuna 88,non digital,0 %,$0
3,NORESTE,DIV MONTERREY,4210,150 SAN NICOLAS DE LOS GARZA,RT001,200000613,Depósito La Victoria 37,non digital,0 %,$0
4,NORESTE,DIV MONTERREY,4210,150 SAN NICOLAS DE LOS GARZA,RT001,200000616,Mercadito Santa Cruz 42,non digital,0 %,$0


In [8]:
#Se verifica la consistencia en las altas de ambas tablas

query="""
SELECT s.ruta_sap, s.cliente_id, s.nombre_cliente
FROM status AS s
LEFT JOIN registros AS r
    ON s.cliente_id  = r.nip_cliente
WHERE r.status_registro IS NULL
"""

pd.read_sql(query,conn)

,ruta_sap,cliente_id,nombre_cliente
0,RT001,200002030,Bodega Del Norte 50


Se identificó un cliente en la tabla de status que no está registrado, lo que figura una inconsistencia en el alta del cliente.

In [9]:
# Se obtiene la absistencia de retención de la aplicación

query="""
SELECT s.ruta_sap, s.cliente_id, s.nombre_cliente, r.status_registro,s."categoría", s."%_venta_digital"
FROM status AS s
LEFT JOIN registros AS r
    ON s.cliente_id  = r.nip_cliente
WHERE r.status_registro = 'registrado' AND s."categoría" = 'non digital'
"""

df=pd.read_sql(query,conn)
df.to_csv(r"../Data_processed_anon/clientes_digital_adoption.csv", index=False)
df

,ruta_sap,cliente_id,nombre_cliente,status_registro,categoría,%_venta_digital
0,RT001,200000018,Abarrotes El Progreso 72,registrado,non digital,0 %
1,RT001,200000324,Minisuper La Esperanza 85,registrado,non digital,0 %
2,RT001,200000612,Depósito La Fortuna 88,registrado,non digital,0 %
3,RT001,200000616,Mercadito Santa Cruz 42,registrado,non digital,0 %
4,RT001,200001691,Bodega La Perla 43,registrado,non digital,0 %
...,...,...,...,...,...,...
702,RT022,200000678,Abarrotes El Progreso 57,registrado,non digital,0 %
703,RT022,200000685,Abarrotes La Fortuna 10,registrado,non digital,0 %
704,RT022,200001599,Comercial El Sol 52,registrado,non digital,0 %
705,RT022,200001709,Tienda El Roble 14,registrado,non digital,0 %


In [10]:

query="""
WITH JOINs AS(
    SELECT s.ruta_sap, s.cliente_id, s.nombre_cliente, r.status_registro,s."categoría", s."%_venta_digital"
    FROM status AS s
    LEFT JOIN registros AS r
        ON s.cliente_id  = r.nip_cliente
    )

SELECT ruta_sap, COUNT (1) AS num_clientes
FROM JOINs
WHERE status_registro = 'registrado' AND "categoría" = 'non digital'
GROUP BY ruta_sap
ORDER BY num_clientes DESC
"""

pd.read_sql(query,conn)

,ruta_sap,num_clientes
0,RT019,70
1,RT011,55
2,RT014,51
3,RT004,49
4,RT002,48
5,RT008,47
6,RT006,44
7,RT003,39
8,RT012,37
9,RT009,37


In [11]:
query="""
WITH JOINs AS(
    SELECT s.ruta_sap, s.cliente_id, s.nombre_cliente, r.status_registro,s."categoría", s."%_venta_digital"
    FROM status AS s
    LEFT JOIN registros AS r
        ON s.cliente_id  = r.nip_cliente
    )

SELECT ruta_sap, COUNT (1) AS num_clientes
FROM JOINs
WHERE status_registro = 'no registrado'
GROUP BY ruta_sap
ORDER BY num_clientes DESC
"""

pd.read_sql(query,conn)

,ruta_sap,num_clientes
0,RT011,61
1,RT015,47
2,RT009,39
3,RT004,33
4,RT008,30
5,RT021,29
6,RT003,29
7,RT002,26
8,RT018,24
9,RT014,23


In [12]:
query="""
WITH JOINs AS(
    SELECT s.ruta_sap, s.cliente_id, s.nombre_cliente, r.status_registro,s."categoría", s."%_venta_digital"
    FROM status AS s
    LEFT JOIN registros AS r
        ON s.cliente_id  = r.nip_cliente
    ), clientes_registros AS(
    SELECT ruta_sap,COUNT (1) AS num_clientes_registro
    FROM JOINs
    WHERE status_registro = 'no registrado'
    GROUP BY ruta_sap
), clientes_adopción_digital AS(
    SELECT ruta_sap, COUNT (1) AS num_clientes_status
    FROM JOINs
    WHERE status_registro = 'registrado' AND "categoría" = 'non digital'
    GROUP BY ruta_sap
)

SELECT cr.ruta_sap, cr.num_clientes_registro, cs.num_clientes_status
FROM clientes_adopción_digital AS cs
LEFT JOIN clientes_registros AS cr
    ON cr.ruta_sap=cs.ruta_sap
ORDER BY num_clientes_registro DESC, num_clientes_status DESC
"""

df=pd.read_sql(query,conn)
df.to_csv(r"../Data_processed_anon/clientes_digital_adoption_register.csv", index=False)
df


,ruta_sap,num_clientes_registro,num_clientes_status
0,RT011,61,55
1,RT015,47,33
2,RT009,39,37
3,RT004,33,49
4,RT008,30,47
5,RT003,29,39
6,RT021,29,33
7,RT002,26,48
8,RT018,24,27
9,RT014,23,51


In [13]:
query= """WITH base AS (
    SELECT 
        s.ruta_sap,
        s.cliente_id,
        r.status_registro,
        s."categoría"
    FROM status AS s
    LEFT JOIN registros AS r
        ON s.cliente_id = r.nip_cliente
    WHERE r.status_registro IS NOT NULL
),

universo AS (
    SELECT 
        ruta_sap,
        COUNT(DISTINCT cliente_id) AS total_clientes
    FROM base
    GROUP BY ruta_sap
),

no_registrados AS (
    SELECT 
        ruta_sap,
        COUNT(DISTINCT cliente_id) AS clientes_no_registrados
    FROM base
    WHERE status_registro = 'no registrado'
    GROUP BY ruta_sap
),

registrados_no_digitales AS (
    SELECT 
        ruta_sap,
        COUNT(DISTINCT cliente_id) AS clientes_reg_no_digital
    FROM base
    WHERE status_registro = 'registrado' AND "categoría" = 'non digital'
    GROUP BY ruta_sap
),  
hibridos AS(
    SELECT 
        ruta_sap,
        COUNT(DISTINCT cliente_id) AS clientes_hibridos
    FROM base
    WHERE "categoría" = 'hibrido'
    GROUP BY ruta_sap
),
fully AS(
    SELECT 
        ruta_sap,
        COUNT(DISTINCT cliente_id) AS clientes_fully
    FROM base
    WHERE "categoría" = 'fully'
    GROUP BY ruta_sap
)

SELECT 
    u.ruta_sap,
    u.total_clientes,
    COALESCE(nr.clientes_no_registrados, 0) AS clientes_no_registrados,
    COALESCE(rnd.clientes_reg_no_digital, 0) AS clientes_reg_no_digital,
    COALESCE(h.clientes_hibridos, 0) AS clientes_hibridos,
    COALESCE(f.clientes_fully, 0) AS clientes_fully,

    ROUND(
        100.0 * clientes_reg_no_digital / u.total_clientes,2
    ) AS pct_registrados_no_digitales,
    
    ROUND(
        100.0 * clientes_no_registrados / u.total_clientes,2
    ) AS pct_no_registrados,
    

     ROUND(
        100.0 * clientes_hibridos / u.total_clientes,2
    ) AS pct_hibridos,

    ROUND(
    COALESCE(100.0 * clientes_fully / NULLIF(u.total_clientes, 0), 0),
    2
) AS pct_fully,

-- Y en el score también:
    ROUND(
    COALESCE(
        (100.0 * clientes_reg_no_digital / u.total_clientes * 0.40) + 
        (100.0 * clientes_no_registrados / u.total_clientes * 0.25) + 
        (100.0 * clientes_hibridos / u.total_clientes * 0.20) + 
        ((100 - COALESCE(100.0 * clientes_fully / u.total_clientes, 0)) * 0.15), 0
    ),2
) AS score_prioridad

FROM universo u
LEFT JOIN registrados_no_digitales rnd
    ON u.ruta_sap = rnd.ruta_sap
LEFT JOIN no_registrados nr
    ON u.ruta_sap = nr.ruta_sap
LEFT JOIN hibridos h
    ON u.ruta_sap = h.ruta_sap
LEFT JOIN fully f
    ON u.ruta_sap = f.ruta_sap



ORDER BY score_prioridad DESC
"""
df=pd.read_sql(query,conn)
df.to_csv(r"../Data_processed_anon/clientes_digital_adoption_prioridad.csv", index=False)
df


,ruta_sap,total_clientes,clientes_no_registrados,clientes_reg_no_digital,clientes_hibridos,clientes_fully,pct_registrados_no_digitales,pct_no_registrados,pct_hibridos,pct_fully,score_prioridad
0,RT019,102,19,66,13,4,64.71,18.63,12.75,3.92,47.50
1,RT014,103,23,51,27,2,49.51,22.33,26.21,1.94,45.34
2,RT008,97,29,47,17,4,48.45,29.90,17.53,4.12,44.74
3,RT009,95,39,37,18,1,38.95,41.05,18.95,1.05,44.47
4,RT013,27,12,11,3,1,40.74,44.44,11.11,3.70,44.07
5,RT004,114,33,49,27,5,42.98,28.95,23.68,4.39,43.51
6,RT011,121,55,50,9,7,41.32,45.45,7.44,5.79,43.51
7,RT002,102,26,48,21,7,47.06,25.49,20.59,6.86,43.28
8,RT003,97,29,39,25,4,40.21,29.90,25.77,4.12,43.09
9,RT001,16,5,5,6,0,31.25,31.25,37.50,0.00,42.81


In [14]:
# Se obtiene el universo de clientes

query="""
SELECT s.ruta_sap, s.cliente_id, s.nombre_cliente, r.status_registro,s."categoría", s."%_venta_digital"
FROM status AS s
LEFT JOIN registros AS r
    ON s.cliente_id  = r.nip_cliente
"""

df=pd.read_sql(query,conn)
df.to_csv(r"../Data_processed_anon/universo_clientes.csv", index=False)
df

,ruta_sap,cliente_id,nombre_cliente,status_registro,categoría,%_venta_digital
0,RT001,200000018,Abarrotes El Progreso 72,registrado,non digital,0 %
1,RT001,200000324,Minisuper La Esperanza 85,registrado,non digital,0 %
2,RT001,200000612,Depósito La Fortuna 88,registrado,non digital,0 %
3,RT001,200000613,Depósito La Victoria 37,no registrado,non digital,0 %
4,RT001,200000616,Mercadito Santa Cruz 42,registrado,non digital,0 %
...,...,...,...,...,...,...
1839,RT020,200000568,Frutería Santa Fe 21,registrado,fully,100 %
1840,RT013,200001454,Super La Unión 67,registrado,hibrido,67 %
1841,RT010,200001420,Comercial Los Alamos 87,registrado,fully,80 %
1842,RT021,200001398,Super Los Pinos 68,registrado,fully,71 %


In [15]:
conn.close()
print("Conexión cerrada")

Conexión cerrada


In [16]:

# Cargar tu CSV final con scores
df = pd.read_csv(r"../Data_processed_anon/clientes_digital_adoption_prioridad.csv")

# 1. Total de clientes
total_clientes = df['total_clientes'].sum()
print(f"Total clientes: {total_clientes}")

# 2. Calcular cuánto concentran las Top N rutas
df_sorted = df.sort_values('score_prioridad', ascending=False)

# Top 5
top5_clientes = df_sorted.head(10)['total_clientes'].sum()
pct_top5 = (top5_clientes / total_clientes) * 100
print(f"Top 10 rutas tienen: {top5_clientes} clientes = {pct_top5:.1f}% del total")

# 3. Registrados inactivos
total_reg_inactivos = df['clientes_reg_no_digital'].sum()
pct_reg_inactivos = (total_reg_inactivos / total_clientes) * 100
print(f"Registrados inactivos: {total_reg_inactivos} = {pct_reg_inactivos:.1f}% del total")

total_no_reg = df['clientes_no_registrados'].sum()
total_reg_no_dig = df['clientes_reg_no_digital'].sum()
total_hibridos = df['clientes_hibridos'].sum()
total_fully = df['clientes_fully'].sum()

pct_no_reg = (total_no_reg / total_clientes) * 100
pct_reg_no_dig = (total_reg_no_dig / total_clientes) * 100
pct_hibridos = (total_hibridos / total_clientes) * 100
pct_fully = (total_fully / total_clientes) * 100

print(f"4. Distribución global:")
print(f"   - No registrados: {pct_no_reg:.1f}% ({total_no_reg} clientes)")
print(f"   - Reg. no digitales: {pct_reg_no_dig:.1f}% ({total_reg_no_dig} clientes)")
print(f"   - Híbridos: {pct_hibridos:.1f}% ({total_hibridos} clientes)")
print(f"   - Fully digital: {pct_fully:.1f}% ({total_fully} clientes)")

# ============================================
# Verificación (debe sumar 100%)
# ============================================
suma_pct = pct_no_reg + pct_reg_no_dig + pct_hibridos + pct_fully
print(f"\n✅ Verificación: {suma_pct:.1f}% (debe ser ~100%)")


Total clientes: 1788
Top 10 rutas tienen: 874 clientes = 48.9% del total
Registrados inactivos: 687 = 38.4% del total
4. Distribución global:
   - No registrados: 25.6% (457 clientes)
   - Reg. no digitales: 38.4% (687 clientes)
   - Híbridos: 22.4% (401 clientes)
   - Fully digital: 13.6% (243 clientes)

✅ Verificación: 100.0% (debe ser ~100%)


In [17]:
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from datetime import datetime

def crear_informe_stakeholder(logo_path, output_path):
    """
    Genera informe ejecutivo en PDF para stakeholder
    
    Args:
        logo_path: Ruta al logo de la empresa (ej: 'logo_empresa.png')
        output_path: Nombre del archivo PDF de salida
    """
    
    # Crear documento PDF
    doc = SimpleDocTemplate(output_path, pagesize=letter,
                           rightMargin=0.75*inch, leftMargin=0.75*inch,
                           topMargin=0.75*inch, bottomMargin=0.75*inch)
    
    # Contenedor para elementos del documento
    story = []
    
    # Estilos
    styles = getSampleStyleSheet()
    
    # Estilo personalizado para título
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=24,
        textColor=colors.HexColor('#003366'),
        spaceAfter=30,
        alignment=TA_CENTER,
        fontName='Helvetica-Bold'
    )
    
    # Estilo para subtítulos
    subtitle_style = ParagraphStyle(
        'CustomSubtitle',
        parent=styles['Heading2'],
        fontSize=16,
        textColor=colors.HexColor('#003366'),
        spaceAfter=12,
        spaceBefore=20,
        fontName='Helvetica-Bold'
    )
    
    # Estilo para texto normal
    normal_style = ParagraphStyle(
        'CustomNormal',
        parent=styles['Normal'],
        fontSize=11,
        alignment=TA_JUSTIFY,
        spaceAfter=12,
        leading=14
    )
    
    # Estilo para destacados
    highlight_style = ParagraphStyle(
        'Highlight',
        parent=styles['Normal'],
        fontSize=11,
        textColor=colors.HexColor('#CC0000'),
        fontName='Helvetica-Bold',
        spaceAfter=6
    )
    
    # ==========================================
    # 1. PORTADA
    # ==========================================
    
    # Logo (ajusta el tamaño según tu logo)
    try:
        logo = Image(logo_path, width=2*inch, height=1*inch)
        logo.hAlign = 'CENTER'
        story.append(logo)
        story.append(Spacer(1, 0.3*inch))
    except:
        print("⚠️ No se pudo cargar el logo. Continuando sin logo...")
        story.append(Spacer(1, 0.5*inch))
    
    # Título principal
    story.append(Paragraph("INFORME EJECUTIVO", title_style))
    story.append(Paragraph("Sistema de Priorización de Rutas<br/>Adopción Digital", title_style))
    story.append(Spacer(1, 0.5*inch))
    
    # Información del documento
    info_data = [
        ["Preparado para:", "Ricardo Fernández del Toro"],
        ["Preparado por:", "Felipe de Jesús Luis Rios"],
        ["Fecha:", datetime.now().strftime("%d/%m/%Y")],
        ["Clasificación:", "Interno - Confidencial"]
    ]
    
    info_table = Table(info_data, colWidths=[2*inch, 4*inch])
    info_table.setStyle(TableStyle([
        ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 0), (-1, -1), 11),
        ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
        ('TEXTCOLOR', (0, 0), (0, -1), colors.HexColor('#003366')),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
    ]))
    
    story.append(info_table)
    story.append(PageBreak())
    
    # ==========================================
    # 2. RESUMEN EJECUTIVO
    # ==========================================
    
    story.append(Paragraph("RESUMEN EJECUTIVO", subtitle_style))
    
    story.append(Paragraph(
        "Este documento presenta los resultados del análisis de priorización de rutas comerciales "
        "para la adopción digital de la aplicación de pedidos. Se analizaron <b>20 rutas</b> con un "
        "total de <b>1,788 clientes</b>, desarrollando un sistema de scoring que permite identificar "
        "dónde enfocar recursos para maximizar el retorno de inversión.",
        normal_style
    ))
    
    story.append(Spacer(1, 0.2*inch))
    
    # Hallazgos destacados
    story.append(Paragraph("<b>Hallazgos Clave:</b>", highlight_style))
    
    hallazgos = [
        "• <b>38.4%</b> de clientes registrados NO usan la aplicación → Oportunidad inmediata",
        "• Top 10 rutas concentran <b>48.9%</b> del potencial total",
        "• Rango de scores: 22.40 (ruta digitalizada) - 47.50 (ruta crítica)",
        "• Distribución: 25.6% no registrados | 38.4% reg. inactivos | 22.4% híbridos | 13.6% fully digital"
    ]
    
    for hallazgo in hallazgos:
        story.append(Paragraph(hallazgo, normal_style))
    
    story.append(Spacer(1, 0.3*inch))
    
    # ==========================================
    # 3. PROBLEMA DE NEGOCIO
    # ==========================================
    
    story.append(Paragraph("PROBLEMA DE NEGOCIO", subtitle_style))
    
    story.append(Paragraph(
        "<b>Situación Actual:</b>",
        highlight_style
    ))
    
    problemas = [
        "• 20 rutas comerciales con niveles heterogéneos de adopción digital",
        "• 1,788 clientes en diferentes etapas del proceso de digitalización",
        "• Recursos limitados del equipo comercial",
        "• Ausencia de metodología estructurada para priorizar intervenciones"
    ]
    
    for problema in problemas:
        story.append(Paragraph(problema, normal_style))
    
    story.append(Spacer(1, 0.15*inch))
    
    story.append(Paragraph(
        "<b>Pregunta Estratégica:</b><br/>"
        "¿Dónde debe enfocar el equipo comercial sus esfuerzos para maximizar "
        "la adopción digital de la aplicación de pedidos?",
        normal_style
    ))
    
    story.append(PageBreak())
    
    # ==========================================
    # 4. METODOLOGÍA DE SCORING
    # ==========================================
    
    story.append(Paragraph("METODOLOGÍA DE SCORING", subtitle_style))
    
    story.append(Paragraph(
        "Se desarrolló un sistema de scoring ponderado que clasifica cada ruta según "
        "su potencial de conversión. A mayor score, mayor prioridad de intervención.",
        normal_style
    ))
    
    story.append(Spacer(1, 0.15*inch))
    
    # Tabla de clasificación
    story.append(Paragraph("<b>Clasificación de Clientes:</b>", highlight_style))
    
    clasificacion_data = [
        ["Categoría", "Definición", "% Ventas Digitales"],
        ["No Registrados", "Sin cuenta en aplicación", "0%"],
        ["Registrados No Digitales", "Cuenta creada pero sin uso", "0%"],
        ["Híbridos", "Uso parcial de la aplicación", "1-69%"],
        ["Fully Digital", "Alto uso de la aplicación", "≥70%"]
    ]
    
    clasificacion_table = Table(clasificacion_data, colWidths=[1.8*inch, 2.5*inch, 1.5*inch])
    clasificacion_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#003366')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('GRID', (0, 0), (-1, -1), 1, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ]))
    
    story.append(Spacer(1, 0.15*inch))
    story.append(clasificacion_table)
    story.append(Spacer(1, 0.2*inch))
    
    # Fórmula de scoring
    story.append(Paragraph("<b>Fórmula de Scoring:</b>", highlight_style))
    story.append(Paragraph(
        "<i>Score = (% Reg. No Digitales × 0.40) + (% No Registrados × 0.25) + "
        "(% Híbridos × 0.20) + ((100 - % Fully) × 0.15)</i>",
        normal_style
    ))
    
    story.append(Spacer(1, 0.15*inch))
    
    # Tabla de justificación de pesos
    story.append(Paragraph("<b>Justificación de Pesos:</b>", highlight_style))
    
    pesos_data = [
        ["Factor", "Peso", "Razón Estratégica"],
        ["Registrados No Digitales", "40%", "Quick wins: Ya registrados, solo activar. Conversión rápida"],
        ["No Registrados", "25%", "Volumen de oportunidad. Requiere registro + activación"],
        ["Híbridos", "20%", "Ya usan app parcialmente. Escalables"],
        ["Gap de Fully", "15%", "Métrica de contexto. Indica margen de crecimiento vs. saturación"]
    ]
    
    pesos_table = Table(pesos_data, colWidths=[1.8*inch, 0.7*inch, 4*inch])
    pesos_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#003366')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('ALIGN', (0, 1), (1, -1), 'CENTER'),
        ('ALIGN', (2, 1), (2, -1), 'LEFT'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 10),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 9),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 6),
        ('GRID', (0, 0), (-1, -1), 1, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ]))
    
    story.append(Spacer(1, 0.15*inch))
    story.append(pesos_table)
    
    story.append(PageBreak())
    
    # ==========================================
    # 5. HALLAZGOS PRINCIPALES
    # ==========================================
    
    story.append(Paragraph("HALLAZGOS PRINCIPALES", subtitle_style))
    
    # Hallazgo 1
    story.append(Paragraph("<b>1. Oportunidad Inmediata: Clientes Registrados Inactivos</b>", highlight_style))
    story.append(Paragraph(
        "<b>38.4%</b> de los clientes (686 personas) ya tienen cuenta creada pero NO usan la aplicación. "
        "Estos clientes representan la <b>oportunidad de conversión más rápida</b> ya que superaron "
        "la barrera del registro.",
        normal_style
    ))
    story.append(Spacer(1, 0.15*inch))
    
    # Hallazgo 2
    story.append(Paragraph("<b>2. Concentración de Oportunidad</b>", highlight_style))
    story.append(Paragraph(
        "Las <b>Top 10 rutas concentran 48.9%</b> del potencial total de conversión. "
        "Enfocar recursos en este segmento maximiza el retorno de inversión.",
        normal_style
    ))
    story.append(Spacer(1, 0.15*inch))
    
    # Hallazgo 3
    story.append(Paragraph("<b>3. Dispersión de Scores</b>", highlight_style))
    story.append(Paragraph(
        "El rango de scores va desde <b>22.40</b> (ruta digitalizada) hasta <b>47.50</b> (ruta crítica). "
        "Esta alta dispersión indica que algunas rutas requieren intervención urgente mientras "
        "otras están bien encaminadas.",
        normal_style
    ))
    story.append(Spacer(1, 0.15*inch))
    
    # Hallazgo 4
    story.append(Paragraph("<b>4. Distribución Global del Universo</b>", highlight_style))
    
    distribucion_data = [
        ["Categoría", "% del Total", "Clientes", "Estrategia"],
        ["Registrados No Digitales", "38.4%", "686", "Prioridad 1: Activación"],
        ["No Registrados", "25.6%", "458", "Prioridad 2: Registro"],
        ["Híbridos", "22.4%", "401", "Prioridad 3: Escalar a fully"],
        ["Fully Digital", "13.6%", "243", "Mantenimiento"]
    ]
    
    distribucion_table = Table(distribucion_data, colWidths=[2*inch, 1*inch, 1*inch, 2*inch])
    distribucion_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#003366')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 10),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('GRID', (0, 0), (-1, -1), 1, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ]))
    
    story.append(Spacer(1, 0.15*inch))
    story.append(distribucion_table)
    
    story.append(PageBreak())
    '''
    # ==========================================
    # 6. PLAN DE ACCIÓN RECOMENDADO
    # ==========================================
    
    story.append(Paragraph("PLAN DE ACCIÓN RECOMENDADO", subtitle_style))
    
    # Fase 1
    story.append(Paragraph("<b>Fase 1 (Mes 1-2): Quick Wins</b>", highlight_style))
    story.append(Paragraph("<b>Objetivo:</b> Activar registrados no digitales en Top 10 rutas", normal_style))
    story.append(Paragraph(
        "<b>Acciones:</b><br/>"
        "• Llamadas personalizadas explicando beneficios<br/>"
        "• Demo de 15 minutos de la aplicación<br/>"
        "• Incentivo: 10% descuento en primer pedido digital<br/>"
        "• Soporte técnico dedicado primera semana",
        normal_style
    ))
    story.append(Paragraph("<b>Meta:</b> Activar 40% de registrados inactivos (274 clientes)", normal_style))
    story.append(Spacer(1, 0.15*inch))
    
    # Fase 2
    story.append(Paragraph("<b>Fase 2 (Mes 3-4): Expansión de Registro</b>", highlight_style))
    story.append(Paragraph("<b>Objetivo:</b> Registrar clientes nuevos en rutas prioritarias", normal_style))
    story.append(Paragraph(
        "<b>Acciones:</b><br/>"
        "• Campaña presencial en rutas Top 10<br/>"
        "• Material impreso con código QR para registro rápido<br/>"
        "• Incentivo: Regalo de bienvenida<br/>"
        "• Follow-up a 1 semana post-registro",
        normal_style
    ))
    story.append(Paragraph("<b>Meta:</b> Registrar 30% de no registrados (137 clientes)", normal_style))
    story.append(Spacer(1, 0.15*inch))
    
    # Fase 3
    story.append(Paragraph("<b>Fase 3 (Mes 5-6): Escalamiento de Híbridos</b>", highlight_style))
    story.append(Paragraph("<b>Objetivo:</b> Incrementar % venta digital en clientes híbridos", normal_style))
    story.append(Paragraph(
        "<b>Acciones:</b><br/>"
        "• Análisis individual de barreras (uso del dashboard drill-through)<br/>"
        "• Capacitación en funcionalidades avanzadas<br/>"
        "• Incentivos por volumen digital<br/>"
        "• Resolución de problemas técnicos",
        normal_style
    ))
    story.append(Paragraph("<b>Meta:</b> Mover 25% de híbridos a fully (100 clientes)", normal_style))
    
    story.append(PageBreak())
    '''
    # ==========================================
    # 7. MÉTRICAS DE SEGUIMIENTO
    # ==========================================
    
    story.append(Paragraph("MÉTRICAS DE SEGUIMIENTO", subtitle_style))
    
    story.append(Paragraph(
        "Se recomienda conectar con la base de datos para actualización en tiempo real:",
        normal_style
    ))
    
    metricas_data = [
        ["KPI", "Frecuencia", "Objetivo 6 Meses"],
        ["Score promedio de rutas", "Mensual", "↓ -15% (de 35.2 a 29.9)"],
        ["% Registrados no digitales", "Mensual", "↓ -40% (de 38.4% a 23%)"],
        ["% Fully digital", "Mensual", "↑ +50% (de 13.6% a 20.4%)"],
        ["Rutas en Top 10 rotadas", "Trimestral", "5 rutas digitalizadas"]
    ]
    
    metricas_table = Table(metricas_data, colWidths=[2.2*inch, 1.5*inch, 2.3*inch])
    metricas_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#003366')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 10),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('GRID', (0, 0), (-1, -1), 1, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ]))
    
    story.append(Spacer(1, 0.15*inch))
    story.append(metricas_table)
    
    story.append(Spacer(1, 0.3*inch))
    
    # ==========================================
    # 8. HERRAMIENTAS DESARROLLADAS
    # ==========================================
    
    story.append(Paragraph("HERRAMIENTAS DESARROLLADAS", subtitle_style))
    
    story.append(Paragraph(
        "Se desarrolló un <b>dashboard interactivo en Power BI</b> que permite:",
        normal_style
    ))
    
    herramientas = [
        "• Visualización de todas las rutas ordenadas por prioridad (score)",
        "• KPIs de score máximo, mínimo y promedio",
        "• Top 5 rutas prioritarias con gráfico de barras",
        "• Drill-through: Al hacer clic en una ruta, se muestran los clientes específicos de esa ruta",
        "• Formato condicional (semáforo) para identificación visual rápida",
        "• Distribución global de clientes por categoría"
    ]
    
    for herramienta in herramientas:
        story.append(Paragraph(herramienta, normal_style))
    
    story.append(Spacer(1, 0.15*inch))
    story.append(Paragraph(
        "<b>El dashboard permite al equipo comercial:</b>",
        highlight_style
    ))
    story.append(Paragraph(
        "1. Identificar rápidamente rutas prioritarias<br/>"
        "2. Ver el detalle de clientes dentro de cada ruta<br/>"
        "3. Monitorear el progreso mensualmente",
        normal_style
    ))
    
    story.append(PageBreak())
    
    # ==========================================
    # 9. CONTACTO Y PRÓXIMOS PASOS
    # ==========================================
    
    story.append(Paragraph("CONTACTO", subtitle_style))
    
    story.append(Paragraph(
        "Para preguntas sobre metodología, interpretación de datos o uso del dashboard:",
        normal_style
    ))
    
    story.append(Spacer(1, 0.1*inch))
    
    contacto_data = [
        ["Analista:", "Felipe de Jesús Luis Rios"],
        ["Email:", "felipedejesusluisrios@gmail.com"],
        ["LinkedIn:", "www.linkedin.com/in/felipe-de-jesus-luis-rios-0b1917266"]
    ]
    
    contacto_table = Table(contacto_data, colWidths=[1.5*inch, 4*inch])
    contacto_table.setStyle(TableStyle([
        ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 0), (-1, -1), 11),
        ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ]))
    
    story.append(contacto_table)
    
    story.append(Spacer(1, 0.3*inch))
    
    # ==========================================
    # 10. PIE DE PÁGINA
    # ==========================================
    
    story.append(Spacer(1, 0.5*inch))
    
    story.append(Paragraph("CONFIDENCIALIDAD", subtitle_style))
    story.append(Paragraph(
        "Este documento contiene información comercial sensible y está destinado "
        "exclusivamente para uso interno. No debe ser compartido fuera de la organización "
        "sin autorización expresa.",
        normal_style
    ))
    
    story.append(Spacer(1, 0.3*inch))
    
    # Logo final (opcional)
    try:
        logo_final = Image(logo_path, width=1.5*inch, height=0.75*inch)
        logo_final.hAlign = 'CENTER'
        story.append(logo_final)
    except:
        pass
    
    story.append(Paragraph(
        f"© {datetime.now().year} Bebidas del Norte S.A. (empresa ficticia - proyecto de portafolio).",
        ParagraphStyle('Footer', parent=styles['Normal'], fontSize=9, 
                      textColor=colors.grey, alignment=TA_CENTER)
    ))
    
    # ==========================================
    # GENERAR PDF
    # ==========================================
    
    doc.build(story)
    print(f"✅ Informe generado exitosamente: {output_path}")

# ==========================================
# EJECUTAR
# ==========================================

# Uso:
crear_informe_stakeholder(
    logo_path=None,  # sin logo real: proyecto de portafolio anonimizado
    output_path="../Reportes_anon/Informe_Adopcion_Digital_Anonimizado.pdf"
)

⚠️ No se pudo cargar el logo. Continuando sin logo...
✅ Informe generado exitosamente: ../Reportes_anon/Informe_Adopcion_Digital_Anonimizado.pdf
